# Iris Flower Classification — Multi-Class Classification
### Supervised Machine Learning — Classification
---

## Problem Statement

A botanist wants to automatically identify the **species of an iris flower** based on four physical measurements. Instead of predicting a number (like house price or recovery days), we predict a **category** — which species the flower belongs to.

| Target | Description |
|---|---|
| `setosa` | Species 0 — easy to separate |
| `versicolor` | Species 1 — overlaps slightly with virginica |
| `virginica` | Species 2 — overlaps slightly with versicolor |

**Input Features:** Sepal Length, Sepal Width, Petal Length, Petal Width (all in cm)

---

## Classification vs Regression — The Core Difference

| Aspect | Regression (House Price) | Classification (Iris Species) |
|---|---|---|
| **Output** | A continuous number (e.g., $328,000) | A discrete category (e.g., "setosa") |
| **Output type** | Float | Integer label / class name |
| **Loss function** | Mean Squared Error | Cross-Entropy Loss |
| **Evaluation** | MAE, RMSE, R² | Accuracy, Precision, Recall, F1 |
| **Example algorithms** | LinearRegression | LogisticRegression, KNN, DecisionTree |

> **Java analogy:**
> - Regression = method returning `double` (e.g., `predictPrice()`)
> - Classification = method returning an `enum` or `int` (e.g., `predictSpecies()` returning `Species.SETOSA`)

---

## What We Will Build

We will train **three different classifiers** and compare them:

| Algorithm | How It Works |
|---|---|
| **Logistic Regression** | Learns a linear boundary; outputs probabilities for each class |
| **K-Nearest Neighbors (KNN)** | Finds the k closest training points and votes on the class |
| **Decision Tree** | Learns a flowchart of yes/no rules to separate classes |

### The ML Classification Pipeline

```
Raw Data → Explore → Visualize → Split (Train/Test) → Train Models → Evaluate → Compare → Predict
```

---
## Step 1 — Import Libraries

> Same libraries as before, plus classification-specific tools from sklearn:
> - `LogisticRegression` — the go-to starting classifier (despite the name, it's for classification!)
> - `KNeighborsClassifier` — simple, intuitive, no training phase
> - `DecisionTreeClassifier` — interpretable, learns if/else rules
> - `accuracy_score` — what % did we get right?
> - `confusion_matrix` — which classes did we confuse with each other?
> - `classification_report` — precision, recall, F1 per class

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn — dataset
from sklearn.datasets import load_iris              # famous built-in dataset, no CSV needed

# sklearn — data preparation
from sklearn.model_selection import train_test_split  # split into train/test sets

# sklearn — classifiers (three different algorithms to compare)
from sklearn.linear_model import LogisticRegression        # learns a linear decision boundary
from sklearn.neighbors import KNeighborsClassifier         # classifies by majority vote of k nearest neighbors
from sklearn.tree import DecisionTreeClassifier            # learns a tree of if/else rules
from sklearn.tree import plot_tree                         # visualise the decision tree

# sklearn — evaluation metrics
from sklearn.metrics import (
    accuracy_score,        # % of predictions that were correct
    confusion_matrix,      # table: which classes got confused with which
    classification_report, # precision, recall, F1 per class
    ConfusionMatrixDisplay # helper to plot the confusion matrix as a heatmap
)

print('Libraries imported successfully!')

---
## Step 2 — Load and Explore the Dataset

The Iris dataset is a **classic benchmark** in ML — every textbook uses it. It is built into sklearn so there is no CSV file needed.

| Column | Type | Description |
|---|---|---|
| `sepal length (cm)` | float | Length of the sepal (the green leaf-like part) |
| `sepal width (cm)` | float | Width of the sepal |
| `petal length (cm)` | float | Length of the petal (the colourful part) |
| `petal width (cm)` | float | Width of the petal |
| **`target`** | **int** | **0 = setosa, 1 = versicolor, 2 = virginica** |

> **Key fact:** 150 samples, 3 classes, 50 samples each — perfectly balanced dataset.
> No missing values — so we skip the NaN-handling step entirely here.

In [ ]:
# Load the iris dataset — returns a Bunch object (like a dict with attribute access)
iris = load_iris()

# Convert to a pandas DataFrame so we can explore it like we did in previous notebooks
# iris.data       = the feature matrix (numpy array, shape 150×4)
# iris.target     = the label vector  (numpy array, shape 150,)
# iris.feature_names = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
# iris.target_names  = ['setosa', 'versicolor', 'virginica']

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target                                       # add numeric label (0, 1, 2)
df['species_name'] = df['species'].map({                          # add human-readable name
    0: 'setosa',
    1: 'versicolor',
    2: 'virginica'
})

print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Features: {list(iris.feature_names)}')
print(f'Classes:  {list(iris.target_names)}')
print(f'No missing values: {df.isnull().sum().sum()} nulls\n')
df.head(10)

In [ ]:
# Summary statistics grouped by species — compare the means across classes
# A useful feature for classification will have DIFFERENT means per class
# (because the model needs to tell them apart)
print('Summary statistics per species:')
df.groupby('species_name')[iris.feature_names].mean().round(2)

In [ ]:
# Class distribution — how many samples per class?
# For classification, BALANCED classes are ideal (equal samples per class).
# Imbalanced classes (e.g., 90% one class, 10% another) require special handling.
print('Class distribution:')
print(df['species_name'].value_counts())
print(f'\nAll classes have equal representation: {df["species"].value_counts().std() == 0}')

---
## Step 3 — Exploratory Data Analysis (EDA)

Before training any model, always visualise the data. We want to know:
1. **Which features separate the classes best?** → Good features = separable clusters
2. **Is there overlap between classes?** → Overlap = harder to classify
3. **What are the value ranges?** → Determines whether scaling is needed

> **Rule of thumb:** If you can draw a line (or a few lines) on a scatter plot to separate the classes, a linear classifier like Logistic Regression will do well. If the boundary is curved, tree-based or kernel-based methods may do better.

In [ ]:
# Scatter plot matrix — every feature pair plotted against each other
# The diagonal shows the distribution (histogram) of each feature per class.
# Off-diagonal: scatter plot of feature A vs feature B, coloured by class.
# LOOK FOR: well-separated colour clusters = that pair of features is useful for classification.

colors = {'setosa': 'red', 'versicolor': 'green', 'virginica': 'blue'}

fig, axes = plt.subplots(4, 4, figsize=(14, 12))
features = iris.feature_names

for row_idx, y_feat in enumerate(features):
    for col_idx, x_feat in enumerate(features):
        ax = axes[row_idx, col_idx]
        
        if row_idx == col_idx:
            # Diagonal: histogram per class (shows value range and distribution)
            for species, color in colors.items():
                subset = df[df['species_name'] == species]
                ax.hist(subset[x_feat], alpha=0.6, color=color, bins=15, label=species)
            ax.set_title(x_feat.replace(' (cm)', ''), fontsize=8, fontweight='bold')
        else:
            # Off-diagonal: scatter plot coloured by species
            for species, color in colors.items():
                subset = df[df['species_name'] == species]
                ax.scatter(subset[x_feat], subset[y_feat],
                           c=color, alpha=0.6, s=15, label=species)
        
        # Only label outer axes to avoid clutter
        if row_idx == 3: ax.set_xlabel(x_feat.replace(' (cm)', ''), fontsize=7)
        if col_idx == 0: ax.set_ylabel(y_feat.replace(' (cm)', ''), fontsize=7)
        ax.tick_params(labelsize=6)

# Add a shared legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=s)
           for s, c in colors.items()]
fig.legend(handles=handles, loc='upper right', title='Species', fontsize=9)
plt.suptitle('Scatter Plot Matrix — All Feature Pairs by Species', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('OBSERVATION: Petal length and petal width separate the classes most clearly.')
print('Setosa (red) is always isolated. Versicolor and Virginica overlap slightly.')

In [ ]:
# Box plots — distribution of each feature per class side by side
# Box plot anatomy:
#   - The box spans Q1 (25th percentile) to Q3 (75th percentile)
#   - The line inside the box = median (50th percentile)
#   - Whiskers extend to 1.5× IQR
#   - Dots beyond whiskers = potential outliers
# WHAT TO LOOK FOR: boxes that DON'T overlap between classes = strong feature for separation

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
species_order = ['setosa', 'versicolor', 'virginica']
palette = {'setosa': 'salmon', 'versicolor': 'lightgreen', 'virginica': 'cornflowerblue'}

for i, feature in enumerate(iris.feature_names):
    sns.boxplot(
        data=df, x='species_name', y=feature,
        order=species_order, palette=palette, ax=axes[i]
    )
    axes[i].set_title(feature, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Feature Distributions per Species (Box Plots)', fontsize=13)
plt.tight_layout()
plt.show()

print('KEY INSIGHT:')
print('  - Petal length: setosa ~1.5 cm, versicolor ~4.3 cm, virginica ~5.6 cm → very separable!')
print('  - Sepal width: boxes overlap heavily → weaker feature for classification')

In [ ]:
# Correlation heatmap — same as regression notebooks
# For classification this tells us: which features move together?
# Highly correlated features carry redundant information (the model doesn't gain much from both).

plt.figure(figsize=(8, 6))
numeric_df = df[iris.feature_names]  # only the 4 measurement columns
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print('OBSERVATION: Petal length and petal width are strongly correlated (r ≈ 0.96).')
print('They carry almost the same information — using both is slightly redundant.')

---
## Step 4 — Define Features (X) and Target (y)

In classification:
- **X** = all 4 measurement columns (input features)
- **y** = the species label (0, 1, or 2) — this is what the model will predict

```
Java analogy:
  X = method parameters (measurements passed in)
  y = the return value (which species enum to return)
  classifier = the method body that maps inputs → class label
```

> We use all 4 features here, even though petal measurements are stronger.
> Logistic Regression and Decision Tree will naturally learn to weight them accordingly.

In [ ]:
# Separate features and target — identical pattern to regression notebooks
X = iris.data    # numpy array, shape (150, 4) — the 4 measurements
y = iris.target  # numpy array, shape (150,)  — 0, 1, or 2

# Same as: X = df[iris.feature_names].values; y = df['species'].values

print(f'X shape: {X.shape}  →  {X.shape[0]} flowers, {X.shape[1]} measurements each')
print(f'y shape: {y.shape}  →  {y.shape[0]} class labels (0, 1, or 2)')
print(f'\nUnique class labels: {np.unique(y)}  →  {list(iris.target_names)}')
print(f'\nSample rows (first 5):')
for i in range(5):
    print(f'  X[{i}] = {X[i]}  →  y[{i}] = {y[i]} ({iris.target_names[y[i]]})')

---
## Step 5 — Train / Test Split

Same concept as regression: we reserve a portion of data the model will never see during training. We evaluate on this held-out set to measure **generalisation** (how well it handles new, unseen flowers).

### The `stratify=y` parameter — new concept!

Without `stratify`, a random split might accidentally put all setosa samples into training and none into test (especially with small datasets).

```
stratify=y  →  ensures each class has the same proportion in both train and test sets
              (balanced 70/30 per class, not just overall)
```

> **Java analogy:** Like `stratify` is a proportional sampling strategy — instead of a random `Collections.shuffle()`, we ensure each category is evenly represented.

In [ ]:
# Split into 70% training and 30% testing
# stratify=y ensures each class is proportionally represented in BOTH splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,     # 30% for testing → 45 flowers
    random_state=42,   # reproducible split (same split every run)
    stratify=y         # keep class proportions equal in train & test
)

print(f'Training set: {X_train.shape[0]} flowers  (70%)')
print(f'Test set:     {X_test.shape[0]} flowers   (30%)')

# Verify stratification worked — each class should have same % in both sets
print(f'\nClass distribution in training set:')
for i, name in enumerate(iris.target_names):
    count = (y_train == i).sum()
    print(f'  {name:<12}: {count} samples  ({count/len(y_train)*100:.0f}%)')

print(f'\nClass distribution in test set:')
for i, name in enumerate(iris.target_names):
    count = (y_test == i).sum()
    print(f'  {name:<12}: {count} samples  ({count/len(y_test)*100:.0f}%)')

---
## Step 6 — Train Three Classifiers

We will train all three models with the same train/test split so the comparison is fair.

### Algorithm Breakdown

**Logistic Regression**
- Despite the name, it is a classifier (not regression!)
- It learns a linear boundary in feature space
- Outputs a **probability** for each class (via the softmax function)
- Best for: linearly separable data, interpretable results

**K-Nearest Neighbors (KNN)**
- No training phase at all — it just stores the training data
- At prediction time: find the k=5 closest training points, return the majority class
- `k=5` means: "look at 5 neighbours and vote"
- Best for: small datasets, non-linear boundaries

**Decision Tree**
- Learns a binary flowchart: "Is petal length < 2.45? → setosa"
- `max_depth=4` limits tree depth to prevent overfitting
- Best for: interpretable models, non-linear boundaries

> **Java analogy for Decision Tree:** Like a series of `if/else if` statements auto-generated from data:
> ```java
> if (petalLength < 2.45) return SETOSA;
> else if (petalWidth < 1.75) return VERSICOLOR;
> else return VIRGINICA;
> ```

In [ ]:
# Define all three classifiers in a dict so we can loop over them cleanly
classifiers = {
    'Logistic Regression': LogisticRegression(
        max_iter=200,   # increase from default 100 because iris data needs a few more iterations
        random_state=42
    ),
    'K-Nearest Neighbors': KNeighborsClassifier(
        n_neighbors=5   # use 5 nearest neighbours to vote — classic default
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=4,    # limit tree depth → prevents memorising every training sample (overfitting)
        random_state=42
    )
}

# Train all classifiers and store results
results = {}

print(f'{"Classifier":<25} {"Train Acc":>10} {"Test Acc":>10}')
print('-' * 48)

for name, clf in classifiers.items():
    # .fit() — the classifier learns the decision boundary from training data
    # For Logistic Regression: finds weights to separate classes
    # For KNN: just stores X_train and y_train (no real computation yet)
    # For Decision Tree: builds the if/else tree recursively
    clf.fit(X_train, y_train)

    # Predict on both train and test sets
    y_pred_train = clf.predict(X_train)
    y_pred_test  = clf.predict(X_test)

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc  = accuracy_score(y_test,  y_pred_test)

    results[name] = {
        'model': clf,
        'y_pred': y_pred_test,
        'train_acc': train_acc,
        'test_acc': test_acc
    }

    print(f'{name:<25} {train_acc:>10.2%} {test_acc:>10.2%}')

print()
print('NOTE: If train accuracy >> test accuracy → overfitting (model memorised training data).')
print('      Train ≈ test accuracy → good generalisation.')

---
## Step 7 — Evaluate: Confusion Matrix

Accuracy alone doesn't tell the full story. The **confusion matrix** shows exactly which classes the model confuses with each other.

```
                  PREDICTED
                  Setosa  Versicolor  Virginica
ACTUAL  Setosa    [ 15         0          0  ]   ← All setosa correctly identified
        Versicolor[  0        14          1  ]   ← 1 versicolor misclassified as virginica
        Virginica [  0         1         14  ]   ← 1 virginica misclassified as versicolor
```

- **Diagonal cells** (top-left to bottom-right) = correct predictions
- **Off-diagonal cells** = mistakes (row = what it actually was, column = what the model said)

> **Why this matters:** Accuracy = 44/45 = 97.8%, but the confusion matrix tells you *which* species are being confused. In medical diagnosis, knowing which disease is being confused with which other disease is critical.

In [ ]:
# Plot confusion matrices for all three classifiers side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, data) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, data['y_pred'])

    # ConfusionMatrixDisplay draws a colour-coded heatmap of the confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')

    ax.set_title(f'{name}\nAccuracy: {data["test_acc"]:.2%}', fontsize=11)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices — Which Species Get Confused?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Print raw confusion matrix numbers for each classifier
print('Raw confusion matrix values:')
for name, data in results.items():
    cm = confusion_matrix(y_test, data['y_pred'])
    total_errors = cm.sum() - np.trace(cm)   # total off-diagonal = total misclassifications
    print(f'\n  {name}: {total_errors} misclassification(s) out of {len(y_test)} test samples')

---
## Step 8 — Evaluate: Classification Report (Precision, Recall, F1)

`classification_report` gives us three metrics per class:

| Metric | Formula | Plain English |
|---|---|---|
| **Precision** | TP / (TP + FP) | Of everything the model labelled as "setosa", how many actually were setosa? |
| **Recall** | TP / (TP + FN) | Of all the actual setosas in the test set, how many did the model catch? |
| **F1 Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall — balanced summary |
| **Support** | — | How many actual samples of this class were in the test set |

### When does precision vs recall matter?

| Scenario | Optimise for | Why |
|---|---|---|
| Spam filter | **Precision** | Don't want real emails marked as spam (false positive is costly) |
| Cancer detection | **Recall** | Don't want a sick patient marked as healthy (false negative is costly) |
| Search engine | **Both (F1)** | Want relevant results and don't want to miss important ones |

> **For balanced datasets (like iris):** accuracy ≈ F1, so the difference is small.
> For imbalanced datasets, always use F1/precision/recall instead of accuracy.

In [ ]:
# Print full classification report for each classifier
for name, data in results.items():
    print('=' * 55)
    print(f'  {name}')
    print('=' * 55)
    # classification_report prints precision, recall, f1 per class
    # target_names maps 0→'setosa', 1→'versicolor', 2→'virginica' in the output
    print(classification_report(y_test, data['y_pred'], target_names=iris.target_names))
    print()

---
## Step 9 — Deep Dive: Logistic Regression Probabilities

Unlike regression models which always output a number, classifiers need to make a discrete decision. Logistic Regression first computes a **probability for each class**, then picks the class with the highest probability.

```
Input:  [sepal_l=5.1, sepal_w=3.5, petal_l=1.4, petal_w=0.2]
                        ↓
        [P(setosa)=0.97,  P(versicolor)=0.02,  P(virginica)=0.01]
                        ↓
        argmax → class 0 (setosa)
```

> **Java analogy:** Like a method that returns a `Map<Species, Double>` with probabilities, then picks `Collections.max()`. The final `predict()` is just `argmax` of `predict_proba()`.

In [ ]:
log_reg = results['Logistic Regression']['model']

# predict_proba() returns a 2D array: rows = samples, columns = class probabilities
# Each row sums to 1.0 (they are probabilities)
probs = log_reg.predict_proba(X_test)
preds = log_reg.predict(X_test)

print('Logistic Regression — Probability Output for First 10 Test Samples:')
print()
print(f'  {"#":<4} {"P(setosa)":>12} {"P(versicolor)":>14} {"P(virginica)":>14}  {"Predicted":<14} {"Actual":<14} {"Correct?"}')
print('  ' + '-' * 82)

for i in range(10):
    predicted_name = iris.target_names[preds[i]]
    actual_name    = iris.target_names[y_test[i]]
    correct        = '✓' if preds[i] == y_test[i] else '✗ WRONG'
    print(f'  {i+1:<4} {probs[i,0]:>12.4f} {probs[i,1]:>14.4f} {probs[i,2]:>14.4f}  {predicted_name:<14} {actual_name:<14} {correct}')

print()
print('NOTE: High-confidence predictions → one probability close to 1.0, others near 0.')
print('      Low-confidence predictions → probabilities spread more evenly (model is uncertain).')

In [ ]:
# Logistic Regression coefficients — what the model learned
# Each row = one vs rest classifier; each column = weight for that feature
# Positive coefficient → feature increases probability of that class
# Negative coefficient → feature decreases probability of that class

print('Logistic Regression Learned Coefficients:')
print()
coef_df = pd.DataFrame(
    log_reg.coef_,
    index=iris.target_names,
    columns=iris.feature_names
)
print(coef_df.round(3).to_string())
print()
print('Read as: For setosa, higher "petal length (cm)" strongly DECREASES P(setosa).')
print('         The model uses these weights to push each class\'s probability up or down.')

---
## Step 10 — Decision Boundaries

A **decision boundary** is the line (or curve) in feature space that separates one class from another. Visualising it helps us understand *how* each algorithm separates the classes.

We use only **petal length vs petal width** (the two most powerful features) to draw a 2D plot.

- The **coloured regions** are zones where the classifier predicts a specific class
- The **dots** are the actual data points
- Where regions change colour = the decision boundary

> **Key insight:** Logistic Regression draws straight lines (linear boundary).
> Decision Tree draws axis-aligned rectangles (horizontal/vertical cuts).
> KNN draws irregular, curvy boundaries that closely follow the training data.

In [ ]:
# Use only petal length (index 2) and petal width (index 3) for 2D visualisation
feature_idx = [2, 3]   # petal length, petal width
X_2d       = X[:, feature_idx]
X_train_2d = X_train[:, feature_idx]
X_test_2d  = X_test[:, feature_idx]

# Colour maps for regions and scatter points
region_cmap  = plt.cm.RdYlBu        # background regions
point_colors = ['red', 'gold', 'blue']  # per-class scatter dots

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, data) in zip(axes, results.items()):
    # Re-train the same classifier on 2D features only (for visualisation only)
    clf_2d = type(data['model'])(**(data['model'].get_params()))
    clf_2d.fit(X_train_2d, y_train)

    # Build a dense mesh grid that covers the entire feature space
    x_min, x_max = X_2d[:, 0].min() - 0.3, X_2d[:, 0].max() + 0.3
    y_min, y_max = X_2d[:, 1].min() - 0.3, X_2d[:, 1].max() + 0.3
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),   # 300×300 grid = 90,000 points
        np.linspace(y_min, y_max, 300)
    )

    # Predict class for every grid point → defines the coloured regions
    Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Draw coloured regions
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=region_cmap)
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.5)   # draw the actual boundary lines

    # Overlay the actual training and test data points
    for class_idx, (color, species) in enumerate(zip(point_colors, iris.target_names)):
        # Training points (circles)
        mask_train = y_train == class_idx
        ax.scatter(X_train_2d[mask_train, 0], X_train_2d[mask_train, 1],
                   c=color, edgecolors='k', s=50, alpha=0.7, marker='o')
        # Test points (stars, larger) — these are what we evaluate on
        mask_test = y_test == class_idx
        ax.scatter(X_test_2d[mask_test, 0],  X_test_2d[mask_test, 1],
                   c=color, edgecolors='k', s=120, alpha=0.9, marker='*', label=species)

    acc_2d = accuracy_score(y_test, clf_2d.predict(X_test_2d))
    ax.set_title(f'{name}\n(2D accuracy: {acc_2d:.2%})', fontsize=11)
    ax.set_xlabel('Petal Length (cm)')
    ax.set_ylabel('Petal Width (cm)')
    ax.legend(title='Species', fontsize=8)

plt.suptitle('Decision Boundaries — How Each Classifier Separates the Classes\n(stars = test set, circles = train set)',
             fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 11 — Visualise the Decision Tree

One advantage of a Decision Tree is that it is **fully interpretable** — you can read the tree and understand exactly why it made each decision. This is unlike Logistic Regression weights (which require some interpretation) and KNN (which has no model weights at all).

The tree prints questions like:
```
Is petal width (cm) <= 0.75?
  YES → setosa (class 0)
  NO  → Is petal length (cm) <= 4.95?
          YES → versicolor (class 1)
          NO  → virginica (class 2)
```

In [ ]:
dt_model = results['Decision Tree']['model']

plt.figure(figsize=(20, 8))
plot_tree(
    dt_model,
    feature_names=iris.feature_names,   # label each split with the feature name
    class_names=iris.target_names,       # label leaf nodes with the species name
    filled=True,                         # colour nodes by majority class
    rounded=True,                        # rounded node boxes
    fontsize=10
)
plt.title('Decision Tree — Learned If/Else Rules for Iris Classification', fontsize=14)
plt.tight_layout()
plt.show()

# Print the tree as text as well (easier to read the exact thresholds)
from sklearn.tree import export_text
tree_rules = export_text(dt_model, feature_names=list(iris.feature_names))
print('Decision Tree Rules (text format):')
print(tree_rules)

---
## Step 12 — Compare Classifiers

Now let us summarise how all three algorithms compare on this dataset.

Things to compare:
- **Test accuracy** — how accurate on unseen data?
- **Train vs test gap** — is it overfitting?
- **Interpretability** — can we explain the predictions?
- **Speed** — how fast to train and predict?

In [ ]:
# Bar chart comparison of train vs test accuracy for all three classifiers
clf_names  = list(results.keys())
train_accs = [results[n]['train_acc'] for n in clf_names]
test_accs  = [results[n]['test_acc']  for n in clf_names]

x = np.arange(len(clf_names))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars_train = ax.bar(x - bar_width/2, train_accs, bar_width,
                    label='Train Accuracy', color='steelblue',  edgecolor='k', alpha=0.8)
bars_test  = ax.bar(x + bar_width/2, test_accs,  bar_width,
                    label='Test Accuracy',  color='coral', edgecolor='k', alpha=0.8)

# Annotate each bar with the accuracy value
for bar in bars_train:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=10)
for bar in bars_test:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Classifier')
ax.set_ylabel('Accuracy')
ax.set_title('Train vs Test Accuracy — Classifier Comparison')
ax.set_xticks(x)
ax.set_xticklabels(clf_names)
ax.set_ylim(0.85, 1.05)
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8, label='Perfect accuracy')
ax.legend()
plt.tight_layout()
plt.show()

# Summary table
print(f'\n{"Classifier":<25} {"Train Acc":>10} {"Test Acc":>10} {"Gap":>8}  Notes')
print('-' * 75)
for name in clf_names:
    gap = results[name]['train_acc'] - results[name]['test_acc']
    note = 'possible overfit' if gap > 0.05 else 'good generalisation'
    print(f'{name:<25} {results[name]["train_acc"]:>10.2%} {results[name]["test_acc"]:>10.2%} {gap:>8.2%}  {note}')

---
## Step 13 — Predict on New Flowers

This is the practical goal: given measurements of a **brand new flower not in the dataset**, which species is it?

The pipeline is:
1. Collect measurements (as a dict/list)
2. Format as a 2D array (1 row × 4 columns) — sklearn always expects 2D
3. Call `clf.predict()` → get class label
4. Call `clf.predict_proba()` → get confidence per class

In [ ]:
# Three new flowers with known characteristics chosen to test each species
new_flowers = pd.DataFrame({
    'sepal length (cm)': [5.1,  6.2,  7.7],
    'sepal width (cm)':  [3.8,  2.9,  3.0],
    'petal length (cm)': [1.5,  4.3,  6.1],
    'petal width (cm)':  [0.3,  1.3,  2.3],
})

# Expected: flower 1 → setosa (tiny petals), flower 2 → versicolor, flower 3 → virginica

print('New Flower Predictions:')
print('=' * 70)

# Iterate over each classifier and predict for all 3 new flowers
for clf_name, data in results.items():
    clf = data['model']
    predicted_labels = clf.predict(new_flowers.values)          # shape (3,)
    
    # predict_proba is only available for Logistic Regression and Decision Tree
    # KNN also supports it but it's based on neighbor vote fractions
    has_proba = hasattr(clf, 'predict_proba')
    if has_proba:
        predicted_probs = clf.predict_proba(new_flowers.values)  # shape (3, 3)

    print(f'\n  {clf_name}:')
    for i in range(len(new_flowers)):
        pred_species = iris.target_names[predicted_labels[i]]
        if has_proba:
            confidence = predicted_probs[i].max() * 100
            print(f'    Flower {i+1}: {pred_species:<12}  (confidence: {confidence:.1f}%)')
        else:
            print(f'    Flower {i+1}: {pred_species}')

print()
print('Expected: Flower 1 → setosa | Flower 2 → versicolor | Flower 3 → virginica')

---
## Summary — What We Learned

### The Complete Classification Pipeline

```
1.  LOAD        →  load_iris()  (built-in dataset, no CSV)
2.  EXPLORE     →  .shape, .groupby().mean(), class distribution
3.  VISUALIZE   →  scatter matrix, box plots, correlation heatmap
4.  SPLIT X/y   →  X = features (4 cols), y = class label (0/1/2)
5.  SPLIT       →  train_test_split with stratify=y  (keep class balance)
6.  TRAIN       →  fit 3 classifiers on same X_train, y_train
7.  PREDICT     →  clf.predict(X_test)
8.  EVALUATE    →  accuracy, confusion matrix, classification report
9.  PROBABILITIES→  clf.predict_proba() — confidence per class
10. BOUNDARIES  →  visualise how each algorithm separates classes
11. INSPECT     →  plot_tree() — read the actual learned rules
12. COMPARE     →  train vs test accuracy across all three classifiers
13. PREDICT NEW →  clf.predict(new_flowers)
```

---

### Key Concepts — Classification vs Regression

| Concept | Regression (House Price) | Classification (Iris) |
|---|---|---|
| **Output** | `float` (e.g. 328000.0) | `int` label (e.g. 0 = setosa) |
| **Evaluation** | MAE, RMSE, R² | Accuracy, Precision, Recall, F1 |
| **Main metric** | R² Score (0–1) | Accuracy (0–1) |
| **Probabilities** | Not applicable | `predict_proba()` per class |
| **Boundary** | No boundary — fits a surface | Decision boundary separates classes |

---

### Classification Metrics Quick Reference

| Metric | Formula | When to Use |
|---|---|---|
| **Accuracy** | correct / total | Balanced classes, quick check |
| **Precision** | TP / (TP+FP) | Cost of false positives is high (spam filter) |
| **Recall** | TP / (TP+FN) | Cost of false negatives is high (cancer screen) |
| **F1 Score** | harmonic mean(P, R) | Imbalanced classes, need balanced view |
| **Confusion Matrix** | table of TP/FP/TN/FN | Understand *which* classes are confused |

---

### Algorithm Trade-offs

| Algorithm | Strengths | Weaknesses |
|---|---|---|
| **Logistic Regression** | Fast, interpretable, probabilistic, no overfitting risk | Only linear boundaries |
| **KNN** | Simple, non-linear, no training time | Slow at prediction, sensitive to irrelevant features |
| **Decision Tree** | Interpretable rules, non-linear | Prone to overfitting (use `max_depth`) |

---

### What Could Improve This Model?

1. **Feature engineering** — create ratio features like `petal_area = length × width`
2. **Try ensemble methods** — Random Forest (many trees averaged) → typically beats single tree
3. **Cross-validation** — instead of one 70/30 split, use 5-fold CV for more reliable accuracy estimate
4. **Hyperparameter tuning** — find optimal `n_neighbors` for KNN or `max_depth` for Decision Tree
5. **More data** — 150 samples is tiny; real-world classification datasets have thousands+

---

### Next Steps

> Next notebook: **Unsupervised Learning with K-Means Clustering** —
> What if we don't have labels and need to discover groups automatically?